# xfig_02 — Clinical Threshold Unlock Map

For each task (column) and required AUROC threshold (row): which is the
**first context length** where performance first meets that threshold?

Gray = never reached. Color (viridis_r) = shorter context = easier to achieve.

Idea #2 from `docs/NEW_PLOT_IDEAS.md`.

**Data**: `analysis.csv`, k=all, split=test.

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

# ── Workspace root: auto-detect by looking for final_results/ ─────────────────
def _find_workspace():
    candidate = Path.cwd().resolve()
    for _ in range(10):
        if (candidate / "final_results").exists():
            return candidate
        if candidate.parent == candidate:
            break
        candidate = candidate.parent
    return Path("/Users/boshra/NSRR-workspace").resolve()

WORKSPACE_ROOT = _find_workspace()
NSRR_TOOLS     = WORKSPACE_ROOT / "NSRR-tools"
EXPLORE_DIR    = NSRR_TOOLS / "results" / "paper_figures" / "explore"
FINAL_OUT      = EXPLORE_DIR / "final"
FINAL_OUT.mkdir(parents=True, exist_ok=True)
TABLES_DIR     = NSRR_TOOLS / "results" / "tables"

# Add explore utils to path
_nb_dir = EXPLORE_DIR / "notebooks"
sys.path.insert(0, str(_nb_dir))

from utils.data_explore import (
    set_root, load_analysis, load_analysis_all_k,
    load_heatmap, load_parquets, load_modality_table,
    subject_predictions, subject_correctness_matrix, CONTEXT_TO_MIN, CTX_ORDER,
)
from utils import panels_explore as xp

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import seaborn as sns

set_root(WORKSPACE_ROOT)
mpl.rcParams.update({
    "figure.dpi": 150,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "serif",
    "font.size": 8,
    "axes.labelsize": 7,
})

# ── Constants ──────────────────────────────────────────────────────────────────
MAIN_TASKS = ["sex_binary", "bmi_binary", "age_class",
              "sleep_efficiency_binary", "apnea_binary"]
TASK_LABEL = xp.TASK_LABEL

_ok = (WORKSPACE_ROOT / "final_results").exists()
print(f"WORKSPACE_ROOT : {WORKSPACE_ROOT}")
print(f"final_results/ : {'✓ found' if _ok else '✗ NOT FOUND'}")


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
TASKS      = MAIN_TASKS   # change to include supp tasks if desired
HEAD       = "transformer"
THRESHOLDS = [0.70, 0.75, 0.80, 0.85, 0.90]

df = load_analysis("phase0_v3", split="test", k="all")
print("Loaded analysis.csv:", df.shape, "rows")
print("Heads:", sorted(df["head"].unique()))
print("Tasks:", sorted(df["task"].unique()))

In [ ]:
fig, ax = plt.subplots(figsize=(7.0, 3.5))

xp.threshold_unlock_heatmap(
    ax, df,
    tasks=TASKS,
    head=HEAD,
    thresholds=THRESHOLDS,
)
ax.set_title(
    f"First context length to reach target AUROC ({HEAD})",
    fontsize=8, pad=6,
)
fig.tight_layout()
plt.show()

In [ ]:
# ── Run when figure looks good ──────────────────────────────────
fig.savefig(str(FINAL_OUT / 'xfig_02_threshold_unlock.pdf'), bbox_inches='tight')
fig.savefig(str(FINAL_OUT / 'xfig_02_threshold_unlock.png'), dpi=150, bbox_inches='tight')
print('Saved →', FINAL_OUT / 'xfig_02_threshold_unlock.pdf')